### Imports

In [ ]:
import pandas as pd
from io_utils import read_json, write_json, read_jsonl, write_jsonl
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
import numpy as np 
import re
import plotly.express as px

from dotenv import load_dotenv
load_dotenv()
exp_dir = os.environ['EXP_DIR'] # IMPORTANT: make sure you have a .env file with the absolute path to the experiment directory

### SFT Token Estimation

In [ ]:
from transformers import AutoTokenizer
data_path = "/shared_workspace_mfs/kirill/sft/tool_call_repeat_failure/sft_data/shell/shell-tasks"
df = pd.read_json(data_path + '/train.jsonl', lines=True, orient='records')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B')
df['token_len'] = df['task'].apply(lambda x: len(tokenizer(x)['input_ids']))

In [ ]:
sys_prompt = """
You are an expert Linux shell user. Your goal is to complete the given task.
Given a task, provide your reasoning in a 'thought' process and then provide the one-line shell command to execute as the 'command'.
The shell command can either be a single command or a chain of several commands, as long as the entire command can be executed in a single turn.
Pay attention to the output of the previous commands.
The 'reasoning', while it can be as verbose as you want, should not include newline `\n` characters.
The 'command' should be a valid shell command to input in the terminal.

TASK: {task}

Please reply strictly in the following format:
  #Reasoning#: [reasoning]
  #Command#: [command]
"""
sys_propmt_token_len = len(tokenizer(sys_prompt)['input_ids'])

In [ ]:
example_output_cmd = """Reasoning:
To extract the 'timestamp' field from each line in the log file, we can use 'jq' to parse the JSON. Then, we can sort and deduplicate the timestamps to save only unique ones to the output file.
To count the number of files containing the word 'confidential' in the directory '/home/user/documents', I will use the 'grep' command to search for the word in all text files and then count the matches. The result will be saved to '/tmp/confidential_count.txt'
the log file, we can use 'jq' to parse the JSON. Then, we can sort and deduplicate the timestamps to save only unique ones to the output file.
#Command#:
'jq -r '.timestamp' /var/log/zeus-monitoring/events.log | sort -u > /tmp/unique_timestamps.txt'"""
output_token_len = len(tokenizer(example_output_cmd)['input_ids'])

In [ ]:
print('PER SAMPLE')
print(f'Avg input tokens:{df['token_len'].mean() + sys_propmt_token_len}')
print(f'Max input tokens:{df['token_len'].max() + sys_propmt_token_len}')
print(f'Min input tokens:{df['token_len'].min() + sys_propmt_token_len}')
print(f'Output tokens: {output_token_len}')

x = 7500
print(f'TOTAL {x}')
print(f'Avg input tokens:{x * (df['token_len'].mean() + sys_propmt_token_len)}')
print(f'Max input tokens:{x * (df['token_len'].max() + sys_propmt_token_len)}')
print(f'Min input tokens:{x * (df['token_len'].min() + sys_propmt_token_len)}')
print(f'Output tokens: {x * output_token_len}')

### Gather episodes per log

In [ ]:
# Capture failing traces:
# Loop (counts number of failure episodes)
# 1. For every tool call, check the next tool reply and see if it matches exactly
# 2. check if the next message is a 1) tool call, 2) if it's identical to the last. If yes, this is considered an episode of length 1
#   * 1 episode = tool call -> result -> identical tool call (3 interactions)
# 3. if yes to both, check the next reply and see if it's identical reply as last tool reply. If yes, increase episode number 
# 4. continue until tool chain ends or the tool/tool reply is different from last one

import re

def _is_function_call(msg: str) -> bool:
    """
    Matches everything between the tags (including newlines)
    """
    func_pattern = r'^<function=.*</function>$'
    return re.match(func_pattern, msg, re.DOTALL) is not None
 
def _extract_episodes(traj_log_path: str) -> list[dict]: 
    """
    Extracts repeated tool call episodes
    
    Args:
        traj_log_path (str): path to trajectory .json file
    
    Returns:
        list[dict]: list of extracted episode instances provided as a dict
    """

    traj = read_json(traj_log_path)

    # episode tracking
    episodes = []
    is_episode = False
    episode_buffer = []

    # go through messages and record episodes
    idx = 0
    num_msgs = len(traj['messages'])

    # metadata of the full trajectory
    total_tokens = traj['response']['usage']['total_tokens']

    while idx < num_msgs:
        
        # get current msg
        curr_msg = traj['messages'][idx]['content']
        
        if _is_function_call(curr_msg):                             # tool call?
            if idx + 2 < num_msgs:                                  # is the msg after the next within the traj bound?
                next_usr_msg = traj['messages'][idx+2]['content']
                if _is_function_call(next_usr_msg):                 # is that message a tool call message?
                    if curr_msg == next_usr_msg:                    # is it a failure episode?
                        is_episode = True
                        episode_buffer += traj['messages'][idx:idx+2]  # append first tool-resp messages to 
                        idx += 2                                       # move idx to the next tool call
                    else:
                        # a tool call but not an episode, set the index to this tool call
                        if is_episode:
                            # episode ended at the next possible msg, so append current tool-resp and end the episode
                            ep_start_idx = idx - len(episode_buffer)
                            episode_buffer += traj['messages'][idx:idx+2]
                            len_episode = len(episode_buffer)
                            if ep_start_idx == 1:
                                # if episode starts at the 2nd message, save only 1 previous msg as context
                                episode_context = [traj['messages'][0]]
                                usr_prompt = episode_context[-1]['content']
                            elif ep_start_idx == 0:
                                # if epi. starts at the first msg of traj, have empty context
                                episode_context = ['']
                                usr_prompt = ''
                            else:
                                episode_context = traj['messages'][idx-len_episode:idx-len_episode+2]  # get 2 msgs before episode
                                usr_prompt = episode_context[-1]['content']
                        # episode_context = traj['messages'][idx-len_episode:idx-len_episode+2]   # get 2 msgs before episode
                            # episode_buffer += traj['messages'][idx:idx+2]
                            # print("context=" + str(len(episode_context)))    #DEBUG
                            # print("episode=" + str(len_episode))             #DEBUG
                            # print("idx=" + str(idx))             #DEBUG
                            # episode_context = traj['messages'][idx-len_episode:idx-len_episode+2]   # get 2 msgs before episode
                            episodes.append({
                                "end_idx": idx+1,
                                "traj": episode_buffer,
                                "len": len_episode,
                                "user_prompt": usr_prompt,
                                "tool_call": episode_buffer[0]['content'],
                                "episode_context": episode_context
                            })
                            # reset buffer and episode flags
                            is_episode = False
                            episode_buffer = []
                        idx += 2  
                else:
                    if is_episode:
                        # episode ended at the next possible msg, so append current tool-resp and end the episode
                        ep_start_idx = idx - len(episode_buffer)
                        episode_buffer += traj['messages'][idx:idx+2]
                        len_episode = len(episode_buffer)
                        if ep_start_idx == 1:
                            # if episode starts at the 2nd message, save only 1 previous msg as context
                            episode_context = [traj['messages'][0]]
                            usr_prompt = episode_context[-1]['content']
                        elif ep_start_idx == 0:
                            # if epi. starts at the first msg of traj, have empty context
                            episode_context = ['']
                            usr_prompt = ''
                        else:
                            episode_context = traj['messages'][idx-len_episode:idx-len_episode+2]  # get 2 msgs before episode
                            usr_prompt = episode_context[-1]['content']
                        # episode_context = traj['messages'][idx-len_episode:idx-len_episode+2]   # get 2 msgs before episode
                        # print("context=" + str(len(episode_context)))    #DEBUG
                        # print("episode=" + str(len_episode))             #DEBUG
                        # print("idx=" + str(idx))             #DEBUG
                        episodes.append({
                            "end_idx": idx+1,
                            "traj": episode_buffer,
                            "len": len_episode,
                            "user_prompt": usr_prompt,
                            "tool_call": episode_buffer[0]['content'],
                            "episode_context": episode_context
                        })
                        # reset buffer and episode flags
                        is_episode = False
                        episode_buffer = []
                    idx += 2 # move idx by 2 to bypass the tool-resp calls
            else:
                # next possible tool call is out of bounds, hence can't be an episode
                if is_episode:
                    # TODO: handle case where tool call is last msg
                    ep_start_idx = idx - len(episode_buffer)
                    episode_buffer += traj['messages'][idx:idx+2]
                    len_episode = len(episode_buffer)
                    if ep_start_idx == 1:
                        # if episode starts at the 2nd message, save only 1 previous msg as context
                        episode_context = [traj['messages'][0]]
                        usr_prompt = episode_context[-1]['content']
                    elif ep_start_idx == 0:
                        # if epi. starts at the first msg of traj, have empty context
                        episode_context = ['']
                        usr_prompt = ''
                    else:
                        episode_context = traj['messages'][idx-len_episode:idx-len_episode+2]  # get 2 msgs before episode
                        usr_prompt = episode_context[-1]['content']
                    episodes.append({
                        "end_idx": idx+1,
                        "traj": episode_buffer,
                        "len": len_episode,
                        "user_prompt": episode_context[-1]['content'],
                        "tool_call": episode_buffer[0]['content'],
                        "episode_context": episode_context
                    })
                break
        else: 
            # not a tool call
            # increment index normally
            idx += 1 
    return episodes, num_msgs, total_tokens
        
# EXP_DIR = "<openhand_dir>/llm_completions/astropy__astropy-7336/hosted_vllm_-1752022115.735713.json"
# x = _extract_episodes(EXP_DIR)

#### Extract episodes per experiment (done post trajectory stitching)

In [ ]:
import glob
import os

def _get_full_traj_for_instance(instance_log_path: str) -> str:
    """
    Gets the path to the file with the full trajectory for the instance
    Works by sorting all paths by name, then returning the last one.
    If no files in that directory, returning an empty string
    """
    sorted_paths = sorted(glob.glob(f"{instance_log_path}/*.json"), key=os.path.basename)
    if sorted_paths:
        return sorted_paths[-1]
    else:
        return ''

In [ ]:
def extract_episodes_from_exp(exp_dir: str, save_to_disk: bool = False) -> None:
    """
    Extracts function call incontinuity episodes per OpenHands experiment
    
    Args:
        exp_dir (str): path to the experiment log directory
        save_to_disk (bool): save the per-instance data into the `exp_dir` as "tool_call_incontinuity_per_instance.jsonl"
        
    Returns:
        list[dict]: list of instance-wise logs with the episode data
    """
    # get the list of instance IDs with CallToolx25 or higher (most likely to encounter this issue)
    # UPDATE: checking all files, not just with CallToolx25 or higher
    table_cache_file = read_json(f"{exp_dir}/table_cache.json")
    # id_list = [x['id'] for x in table_cache_file if 'call_tools_times' in x and x['call_tools_times'] >= 25]
    id_list = [x['id'] for x in table_cache_file]
    
    print(f"Found {len(id_list)} potential cases...")
    
    # eval results
    eval_report = read_json(f"{exp_dir}/report.json")
    empty_ids = eval_report["empty_patch_ids"]
    resolved_ids = eval_report["resolved_ids"]
    
    # extract episodes for every instance
    print("processing instances...")
    exp_episodes = []
    for id in id_list:
        # print(f"{id}...")
        instance_traj_path = _get_full_traj_for_instance(f"{exp_dir}/llm_completions/{id}")
        # no trajectories exists for this instance, skip...
        if not instance_traj_path:
            continue
        episode_list, traj_len, total_tokens = _extract_episodes(instance_traj_path)

        if episode_list:
            # if the instance has episodes, save them into exp_episodes
            # print("episodes found!")
            exp_episodes.append({
                "exp_name": exp_dir.split('\n')[-1],
                "instance_id": id,
                "total_traj_len": traj_len,
                "total_tokens": total_tokens,
                "empty_patch": id in empty_ids,
                "resolved": id in resolved_ids,
                "episodes": episode_list
            })
    
    print("saving...")
    
    # save exp_episodes if asked
    if save_to_disk:
        write_jsonl(fpath=f"{exp_dir}/tool_call_incontinuity_per_instance_all.jsonl",
                    data=exp_episodes)
    
    return exp_episodes

# for testing
# x = extract_episodes_from_exp("<openhands_dir>")

### Post-processing: converting per-instance into per-episode data

In [ ]:
import re

def _extract_command(raw_msg: str, instance_id: str = None) -> str:
    """Extracts single command between <parameter> tags"""
    command_pattern = r"<parameter=command>(.*?)</parameter>"
    result = re.findall(command_pattern, raw_msg, re.DOTALL)
    if not result:
        return "<empty>"
    return result[0].strip()


# DEPRECATED: only extracted `execute_bash` tool response
# def _extract_tool_reply(raw_msg: str, instance_id: str = None) -> str:
#     """Extracts single tool response"""
#     # NOTE: this reply might not fit all types of tools!
#     command_pattern = r"EXECUTION RESULT of \[execute_bash\]:\n(.*?)\n\[The command completed with exit code"
#     result = re.findall(command_pattern, raw_msg, re.DOTALL)
#     if not result:
#         # if no output of command found, return <empty> string
#         return "<empty>"
#     return result[0].strip()

def _extract_tool_reply(raw_msg: str, instance_id: str = None) -> str:
    """Extracts single tool response"""

    if "execute_bash" in raw_msg:
        command_pattern = r"EXECUTION RESULT of \[execute_bash\]:\n(.*?)\n\[The command completed with exit code"
        result = re.findall(command_pattern, raw_msg, re.DOTALL)
        if not result:
            # if no output of command found, return <empty> string
            return "<empty>"
        return result[0].strip()
    else:
        return raw_msg.split(']:\n',1)[-1]

def _extract_bash_cmds(bash_string:str):
    """
    Extracts bash commands, separated by &&
    """
    commands = re.findall(r"^\s*(\S+)|(?:&&|\||;)\s*(\S+)", bash_string)
    commands = [cmd for tup in commands for cmd in tup if cmd]
    # return commands
    if "python" in bash_string:
        if len(commands) > 1:
            return commands[0:2]
        else:
            return [commands[0]]
    else:
        return commands

def _extract_oh_tool(tool_call:str) -> str: 
    oh_func_match = re.search(r"<function=([^>]+)", tool_call)
    if oh_func_match:
        oh_tool = oh_func_match.group(1)
    else:
        oh_tool = 'unknown'
    return oh_tool

def _extract_tool_success(tool_resp:str) -> int:
    """
    Extract tool use success code;
    0 = success, 1 = failure, -1 = unknown, rest = given by tool
    """
    if 'str_replace_editor' in tool_resp:
        return '1' if "ERROR" in tool_resp else '0'
    elif "execute_bash" in tool_resp:
        tool_result_code = tool_resp.split('\n')[-1][-2]
        return tool_result_code
    else:
        return -1

In [ ]:
def convert_to_episode_wise_data(data: list[dict], save_to_disk: bool = False, save_path: str = None) -> list[dict]:
    """
    Converts from instance-wise data to episode-wise data
    
    Args:
        data (str): dict with per-instance data from "tool_call_incontinuity_episodes.jsonl" file
    
    Returns:
        list[dict]: list of dicts containing episode data
    """
    episode_data = []

    for sample in data:
        for episode in sample['episodes']:
            sample_data = {}
            # instance global data
            sample_data['instance_id'] = sample['instance_id']
            sample_data['traj_len'] = sample['total_traj_len']
            sample_data['empty_patch'] = sample['empty_patch']
            sample_data['resolved'] = sample['resolved']
            sample_data['total_tokens'] = sample['total_tokens']
            
            # episode data
            sample_data['ep_traj'] = episode['traj']
            sample_data['ep_len'] = episode['len']
            sample_data['ep_start_idx'] = episode['end_idx'] - episode['len']
            sample_data['ep_prompt'] = episode['user_prompt']
            sample_data['ep_context'] = episode['episode_context']
            
            # ...extract tool call and response
            # collect all calls and resp
            call = [_extract_command(x['content']) for x in episode['traj'] if x['role'] == 'assistant']
            resp = [_extract_tool_reply(x['content']) for x in episode['traj'] if x['role'] == 'user']
            # since each episode has unique command and response, reduce to single str for both
            unique_comm = set(call).pop()
            unique_resp = set(resp).pop()
            
            sample_data['ep_tool_call'] = unique_comm
            sample_data['ep_tool_resp'] = unique_resp
            sample_data['ep_tool_result'] = _extract_tool_success(tool_resp=episode['traj'][1]['content'])
            sample_data['ep_tool_bin'] = unique_comm.split(' ')[0]

            # get oh_tool and list of bash commands
            # oh_tool should be unique for the episode
            sample_data['ep_oh_tools'] = set([_extract_oh_tool(x['content']) for x in episode['traj'] if x['role'] == 'assistant']).pop()
            sample_data['bash_cmds'] = _extract_bash_cmds(unique_comm)
            
            # append to global list
            episode_data.append(sample_data)
    
    # save exp_episodes if asked
    if save_to_disk and save_path:
        write_jsonl(fpath=f"{save_path}/tool_call_incontinuity_episodes_all.jsonl",
                    data=episode_data)
    
    return episode_data

# for testing
# data_dir = "<openhands_path>/tool_call_incontinuity_episodes.jsonl"
# data = read_jsonl(data_dir)
# x = convert_to_episode_wise_data(data)

### Main script: extract episodes for every experiment in "oh-traces"

In [ ]:
# go through all experiments
import glob
import tqdm

data_paths = []
exp_root_dir = os.environ['EXP_DIR']

for exp_dir in tqdm.tqdm(glob.glob(exp_root_dir + "/*")):
    if any(ext in exp_dir for ext in ['.jsonl', '.html']):        # ignore any data files here
        continue
    print(f"------ processing {exp_dir.split('/')[-1]} ------")
    
    # extract episodes
    inst_data = extract_episodes_from_exp(exp_dir=exp_dir, save_to_disk=True)
    
    # convert to per-episode data, and save into a .jsonl
    epi_data = convert_to_episode_wise_data(data=inst_data, save_to_disk=True, save_path=exp_dir)

### Tool usage frequency

Taken from the system prompt

In [ ]:
# go through all experiments and get all tool calls
import glob
import tqdm
import re
import copy

data_paths = []
exp_root_dir = os.environ['EXP_DIR']

global_data = []
for exp_dir in tqdm.tqdm(glob.glob(exp_root_dir + "/*")):
    exp_name = exp_dir.split('/')[-1]
    if any(ext in exp_name for ext in ['.jsonl', '.html']):        # ignore any data files here
        continue
    print(f"------ processing {exp_name} ------")

    # get eval report
    eval_report = read_json(f"{exp_dir}/report.json")
    empty_ids = eval_report["empty_patch_ids"]
    resolved_ids = eval_report["resolved_ids"]

    # go through every instance
    inst_dict = {}
    for instance_dir in glob.glob(exp_dir + "/llm_completions/*"):

        # instance id
        id = instance_dir.split('/')[-1].split(".json")[0]

        # full traj json file
        full_traj_path = _get_full_traj_for_instance(instance_dir)
        if not full_traj_path:
            print(f"No paths for {id}!..")
            continue

        # load data
        # Added this exception as sometimes internal results might be missing the trajectories
        try:
            data = read_json(full_traj_path)
        except Exception as e:
            print(f"No traj. paths for {id} found, skipping...")
            continue

        # go through every message, check if function, and extract the values
        for full_msg in data['messages']:
            info_dict = {}

            # extract msg
            msg = full_msg['content']

            # skip if not tool call
            if not _is_function_call(msg):
                continue

            # get OpenHands tool name
            oh_func_match = re.search(r"<function=([^>]+)", msg)
            if oh_func_match:
                oh_tool = oh_func_match.group(1)
            else:
                oh_tool = 'unknown'

            # get the command and append to the OH tool that called it
            cmd = _extract_command(msg)

            if oh_tool == "execute_bash":
                bash_cmds = _extract_bash_cmds(cmd)
            else:
                bash_cmds = None

            # save to global list
            global_data.append({
                "exp_name": exp_name,
                "sample_id": id,
                "oh_tool": oh_tool,
                "cmd": cmd,
                "first_cmd": bash_cmds[0] if bash_cmds else None,
                "bash_cmds": bash_cmds,
                "resolved": id in resolved_ids,
                "empty_patch": id in empty_ids
            })

# # save to top dir
# write_jsonl(f"{exp_root_dir}/all_exp_tool_calls.jsonl", global_data)

In [ ]:
def _load_all_tool_ep_failure_data(exp_dir: str, save_to_disk: bool = False):
    first = True
    data = None
    for path in glob.glob(f"{exp_dir}/*"):
        if any(ext in path for ext in ['.jsonl', '.html']):        # ignore any data files here
            continue
        fpath = f"{path}/tool_call_incontinuity_episodes_all.jsonl"
        df = pd.DataFrame(read_jsonl(fpath))
        exp_name = path.split('/')[-1]
        df['exp_name'] = exp_name
        if first:
            data = df
            first = False
        else:
            data = pd.concat([data, df], axis=0)

    # save
    if save_to_disk:
        data.to_json(f"{exp_dir}/tool_call_discontinuity_all_exp.jsonl", orient="records", lines=True)

    return data

# df = _load_all_tool_ep_failure_data(os.environ['EXP_DIR'], save_to_disk=True)

### Data and Plot analysis

In [ ]:
# Master code for plotting everything with interactive HTML with plotly
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

exp_path = os.environ['EXP_DIR']
exp_name = exp_path.split('/')[-1]
figs_all = []
figs_fail = []

# ------- TOTAL tool frequency ------------
tool_call_path = f"{exp_path}/all_exp_tool_calls.jsonl"
tool_call_freq_df = pd.read_json(tool_call_path, orient="records", lines=True)

fig1 = px.histogram(tool_call_freq_df, x="oh_tool",
                   title="Histogram of OpenHands Tool Calls",
                   labels={
                    'oh_tool': "OpenHands Tool"
                   })
fig1.show()
figs_all.append(fig1)

fig2 = px.histogram(tool_call_freq_df, x="first_cmd",
                   title="Histogram of Tool Calls (First Command)",
                   labels={
                    'oh_tool': "Tool"
                   })
fig2.show()
figs_all.append(fig2)

def _collect_bash_cmds(df):
    bash_cmds = []
    for cmds in df['bash_cmds'].to_list():
        if cmds:
            bash_cmds += cmds
    return bash_cmds
fig3 = px.histogram(x=_collect_bash_cmds(tool_call_freq_df),
                   title="Histogram of All Tool Calls",
                   labels={
                    'x': "Tool"
                   })

fig3.show()
figs_all.append(fig3)


def _get_tools_by_resolved(df):
    resolved_df = df[df['resolved'] == True]
    unresolved_df = df[df['resolved'] == False]

    return _collect_bash_cmds(resolved_df), _collect_bash_cmds(unresolved_df)
resolved_cmds, unresolved_cmds = _get_tools_by_resolved(tool_call_freq_df)

fig4 = go.Figure(data=[go.Histogram(x=resolved_cmds, name="Resolved Samples", visible=True, opacity=0.6),
                      go.Histogram(x=unresolved_cmds, name="Unresolved Samples", visible=True, opacity=0.6)])
fig4.update_layout(
    title="Tool commands",
    xaxis_title="Tools commands",
    yaxis_title="Frequency",
    bargap=0.2,
    barmode="overlay"
)
fig4.show()
figs_all.append(fig4)

# ----- Tool Discontinuity Episode Info ---------------
df = pd.read_json(f'{exp_path}/tool_call_discontinuity_all_exp.jsonl', orient="records", lines=True)

# OpenHands Tool Histogram resolved vs. unresolved
fig11 = go.Figure(data=[go.Histogram(x=df[df['resolved']==True]['ep_oh_tools'], name="Resolved Samples", visible=True, opacity=0.6),
                      go.Histogram(x=df[df['resolved']==False]['ep_oh_tools'], name="Unresolved Samples", visible=True, opacity=0.6)])
fig11.update_layout(
    title="OpenHands Tool commands",
    xaxis_title="Tools commands",
    yaxis_title="Frequency",
    bargap=0.2,
    barmode="overlay"
)
fig11.show()
figs_fail.append(fig11)

# Bash tools Histogram resolved vs. unresolved
fail_resolved_cmds, fail_unresolved_cmds = _get_tools_by_resolved(df)
fig12 = go.Figure(data=[go.Histogram(x=fail_resolved_cmds, name="Resolved Samples", visible=True, opacity=0.6),
                      go.Histogram(x=fail_unresolved_cmds, name="Unresolved Samples", visible=True, opacity=0.6),])
fig12.update_layout(
    title="Tool commands",
    xaxis_title="Tools commands",
    yaxis_title="Frequency",
    bargap=0.2,
    barmode="overlay"
)
fig12.show()
figs_fail.append(fig12)

# trajectory lengths
fig13 = go.Figure(data=[go.Histogram(x=df[df['resolved']==True]['traj_len'], name="Resolved Samples", visible=True, opacity=0.6),
                      go.Histogram(x=df[df['resolved']==False]['traj_len'], name="Unresolved Samples", visible=True, opacity=0.6),])
fig13.update_layout(
    title="Trajectory lengths",
    xaxis_title="Tools commands",
    yaxis_title="Frequency",
    bargap=0.2,
    barmode="overlay"
)
fig13.show()
figs_fail.append(fig13)

# episode lengths
fig14 = px.histogram(df, x="ep_len",
                   title="Histogram of Fail Episode Lengths",
                   labels={
                    'traj_len': "Length of an fail episode"
                   })
fig14.show()
figs_fail.append(fig14)

# ep start id, length, and experiment name scatter plot
df['norm_ep_start_idx'] = df['ep_start_idx'] / df['traj_len']
fig15 = px.scatter(df, 
                   x='norm_ep_start_idx',
                   y='ep_len',
                   color='exp_name',
                   labels={
                    'norm_ep_start_idx': 'Starting index of an episode (normalized to trajectory length)',
                    'ep_len': 'Length of an episode',
                   }
                   )
fig15.show()
figs_fail.append(fig15)

# Tool call success results
fig16 = go.Figure(data=[
    go.Histogram(x=df[df['ep_oh_tools']=='str_replace_editor']['ep_tool_result'], name="str_replace_editor", visible=True, opacity=0.6),
    go.Histogram(x=df[df['ep_oh_tools']=='execute_bash']['ep_tool_result'], name="execute_bash", visible=True, opacity=0.6),
    go.Histogram(x=df[df['ep_oh_tools']=='think']['ep_tool_result'], name="think", visible=True, opacity=0.6),
    ])
fig16.update_layout(
    title="Tool use result (sucess vs. fail)",
    xaxis_title="OpenHands Tools",
    yaxis_title="Tool Use Result",
    bargap=0.2,
    barmode="overlay"
)
fig16.add_annotation(x=4, y=4,
            text="0 = suceess, 1 = failure, others = unknown",
            showarrow=False,
            yshift=90)
fig16.show()
figs_fail.append(fig16)

# Tool call success results for resolved vs. non-resolved trajectories
fig17 = go.Figure(data=[
    go.Histogram(x=df[(df['ep_oh_tools']=='str_replace_editor') & (df['resolved'] == True)]['ep_tool_result'], name="str_replace_editor, resolved issue", visible=True, opacity=0.6),
    go.Histogram(x=df[(df['ep_oh_tools']=='str_replace_editor') & (df['resolved'] == False)]['ep_tool_result'], name="str_replace_editor, unresolved issue", visible=True, opacity=0.6),
    go.Histogram(x=df[(df['ep_oh_tools']=='execute_bash') & (df['resolved'] == True)]['ep_tool_result'], name="execute_bash, resolved issue", visible=True, opacity=0.6),
    go.Histogram(x=df[(df['ep_oh_tools']=='execute_bash') & (df['resolved'] == False)]['ep_tool_result'], name="execute_bash, unresolved issue", visible=True, opacity=0.6),
    ])
fig17.update_layout(
    title="Tool use result (sucess vs. fail) of resolved/unresolved patches",
    xaxis_title="OpenHands Tools",
    yaxis_title="Tool Use Result",
    bargap=0.2,
    barmode="overlay"
)
fig17.add_annotation(x=4, y=4,
            text="0 = suceess, 1 = failure, others = unknown",
            showarrow=False,
            yshift=90)
fig17.show()
figs_fail.append(fig17)

# save to interactive html
# with open(f'{exp_path}/plots.html', 'w') as f:
#     f.write(f"<h1><center>Tool discontinuity experiments: {exp_name}</center></h1>")
#     f.write("<h2><center>Total tool calls for all experiments</center></h2>")
#     for fig in figs_all:
#         f.write(fig.to_html(full_html=False, include_plotlyjs='cdn'))
#     f.write("<h2><center>Tool Discontinuity Data</center></h2>")
#     for fig in figs_fail:
#         f.write(fig.to_html(full_html=False, include_plotlyjs='cdn'))

#### Matplotlib Implementation, deprecated in favor of plotly

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
import numpy as np
import pandas as pd 
import plotly

def plot_heuristics(df: pd.DataFrame, data_path: str) -> None:

    # summary data
    sum_data = {
        "exp_name": data_path.split('/')[-1],  
        "num_instances": len(glob.glob(f"{'/'.join(data_path.split('/')[:-1])}/llm_completions/*")), 
    }
    print(sum_data)
    
    # plot: episodes per command leading to an empty patch
    plt.figure(1)
    df3 = pd.DataFrame()
    df3['total_count'] = df['ep_tool_bin'].value_counts()
    df3 = df3.merge(df[df['empty_patch'] == True]['ep_tool_bin'].value_counts(), on='ep_tool_bin')
    plt.bar(df3.index, df3['count'] / df3['total_count'])
    ax = plt.gca()
    ax.set_title("Num. of episodes per command causing an empty patch")
    ax.set_xlabel("% of episodes leading to empty patch (out of # of total episodes)")
    ax.tick_params(axis='x', rotation=90)
    
    # plot: episode starting index histogram
    plt.figure(2)
    plt.hist(df['ep_start_idx'])
    ax = plt.gca()
    ax.set_title('Histogram of episode starting index in the trajectory')
    ax.set_xlabel('Episode start index')
    ax.grid()
    
    # plot: episode starting index relative to total trajectory lenght
    plt.figure(3)
    plt.hist(df['ep_start_idx'] / df['traj_len'])
    ax = plt.gca()
    ax.set_title('Episode start location relative to total trajectory length')
    ax.set_xlabel('Normalized ep. start index')
    ax.grid()
    
    # plot: hist of total traj lengths
    plt.figure(4)
    plt.hist(df['traj_len'])
    ax = plt.gca()
    ax.set_title('Histogram of trajectory lengths for instances w/ TCIF')
    ax.set_xlabel('Total length of trajectory')
    ax.grid()
    
    # plot: num of episodes per instance
    plt.figure(5)
    x = df['instance_id'].value_counts().to_dict().keys()
    vals = df['instance_id'].value_counts().to_dict().values()
    plt.plot(x, vals)
    ax = plt.gca()
    ax.set_title('Num. of epsiodes per instance')
    ax.set_xlabel('Instance id')
    ax.tick_params(axis='x', rotation=90)
    
    # plot: historgram of tool commands
    # tutorial on centering the bars in histograms: https://stackoverflow.com/questions/27083051/matplotlib-xticks-not-lining-up-with-histogram
    plt.figure(6)
    plt.hist(df['ep_tool_bin'], bins=np.arange(15) - 0.5, align="mid")
    ax = plt.gca()
    ax.set_title('Histogram of tool call commands')
    ax.grid()
    ax.tick_params(axis='x', rotation=90)

In [ ]:
def plot_tool_command_hist(data_paths_or_tuples: list[str] | list[tuple[str, pd.DataFrame]], opacity: int = 0.4) -> None: 
    """
    Plot tool command histogram across one or many experiments
    """
    
    plt.figure()
    
    for data in data_paths_or_tuples:
        if isinstance(data, tuple):
            exp_name = data[0]
            df = data[1]
        else: 
            # import data into a pandas df
            exp_name = data.split('/')[-1]
            data = f"{data}/tool_call_incontinuity_episodes.jsonl"
            df = pd.DataFrame(read_jsonl(data))
         
        bins = np.arange(len(df['ep_tool_bin'].unique()))
        # plt.hist(df['ep_tool_bin'], bins=np.arange(15) - 0.5, align="mid")
        plt.hist(df['ep_tool_bin'], bins=bins - 0.5, label=exp_name, alpha=opacity)
    
    # set global plot stuff
    ax = plt.gca()
    ax.set_title('Histogram of tool call commands')
    ax.grid()
    ax.tick_params(axis='x', rotation=90)
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.4))

In [ ]:
import glob
path_to_logs = os.environ['EXP_DIR']
dpaths = glob.glob(path_to_logs + "/*")

# for combined exp df
df = pd.DataFrame()
start = True
for dpath in dpaths:
    if any(ext in exp_name for ext in ['.jsonl', '.html']):        # ignore any data files here
        continue
    if start:
        df = pd.DataFrame(read_jsonl(f"{dpath}/tool_call_incontinuity_episodes_all.jsonl"))
        start = False
    else:
        df_new = pd.DataFrame(read_jsonl(f"{dpath}/tool_call_incontinuity_episodes_all.jsonl"))
        df = pd.concat([df, df_new], axis=0)
dpaths = [('total', df)]    

plot_tool_command_hist(dpaths, opacity=0.5)

In [ ]:
df_resolved = df[df['resolved'] == True]
df_unresolved = df[df['resolved'] == False]

dpaths = [
    ('resolved', df_resolved),
    ('unresolved', df_unresolved)
        ]
plot_tool_command_hist(dpaths, opacity=0.5)

In [ ]:
df3 = pd.DataFrame()
df3['total_count'] = df['ep_tool_bin'].value_counts()
df3 = df3.merge(df[df['empty_patch'] == True]['ep_tool_bin'].value_counts(), on='ep_tool_bin')
# plt.bar(df3.index, df3['count'] / df3['total_count'])
plt.bar(df3.index, df3['count'])
ax = plt.gca()
ax.set_title("Num. of episodes per command causing an empty patch")
ax.set_xlabel("% of episodes leading to empty patch (out of # of total episodes)")
ax.tick_params(axis='x', rotation=90)

In [ ]:
import matplotlib.pyplot as plt


# Plot: cumulative number of episodes per starting index
df2 = df.copy(deep=True)
# df2 = df2.sort_values(by=['ep_start_idx'], ascending=True)
df2['num_ep_before'] = df2.apply(lambda row: len(df2[df2['ep_start_idx'] < row['ep_start_idx']]))
# df2['num_ep_before'] = df2.apply(lambda row: print(row.dtype))

In [ ]:
plt.hist(df['ep_start_idx'])
ax = plt.gca()
ax.set_title('Histogram of episode starting index in the trajectory')
ax.set_xlabel('Episode start index')
ax.grid()

In [ ]:
plt.hist(df['ep_start_idx'] / df['traj_len'])
ax = plt.gca()
ax.set_title('Episode start location relative to trajectory length')
ax.set_xlabel('Normalized ep. start index')
ax.grid()

In [ ]:
plt.hist(df['traj_len'])
ax = plt.gca()
ax.set_title('Histogram of trajectory lengths for instances w/ TCIF')
ax.set_xlabel('Total length of trajectory')
ax.grid()

In [ ]:
x = df['instance_id'].value_counts().to_dict().keys()
vals = df['instance_id'].value_counts().to_dict().values()
plt.plot(x, vals)
ax = plt.gca()
ax.set_title('Num. of epsiodes per instance')
ax.set_xlabel('Instance id')
ax.tick_params(axis='x', rotation=90)

In [ ]:
# tutorial on centering the bars in histograms: https://stackoverflow.com/questions/27083051/matplotlib-xticks-not-lining-up-with-histogram
plt.hist(df['ep_tool_bin'], bins=np.arange(15) - 0.5, align="mid")
ax = plt.gca()
ax.set_title('Histogram of tool call commands')
ax.grid()
ax.tick_params(axis='x', rotation=90)